In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

customers_df = spark.read \
    .option("header", "true") \
    .csv("Files/raw/customers/customers_master.csv")

print("Raw customers:", customers_df.count())
customers_df.printSchema()


StatementMeta(, 030a17ad-bb76-45df-9df6-4c736de41ce4, 3, Finished, Available, Finished, False)

Raw customers: 500
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- date_joined: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- total_orders: string (nullable = true)
 |-- is_active: string (nullable = true)



In [2]:
def parse_active(val):
    if val is None: return False
    return val.strip().lower() in ("y","yes","1","true")

active_udf = F.udf(parse_active, T.BooleanType())

city_map = {
    "mumbai":"Mumbai","mum":"Mumbai","mmbai":"Mumbai",
    "delhi":"Delhi","new delhi":"Delhi","ncr":"Delhi",
    "bangalore":"Bangalore","bengaluru":"Bangalore","blr":"Bangalore",
    "hyderabad":"Hyderabad","hyd":"Hyderabad",
    "chennai":"Chennai","pune":"Pune",
    "kolkata":"Kolkata","calcutta":"Kolkata",
    "ahmedabad":"Ahmedabad","jaipur":"Jaipur","surat":"Surat"
}
def clean_city(c):
    if c is None: return "Unknown"
    return city_map.get(c.lower().strip(), c.strip().title())

city_udf = F.udf(clean_city, T.StringType())

cleaned_df = customers_df \
    .withColumn("date_joined",   F.to_date(F.col("date_joined"))) \
    .withColumn("total_orders",  F.col("total_orders").cast(T.IntegerType())) \
    .withColumn("is_active",     active_udf(F.col("is_active"))) \
    .withColumn("city",          city_udf(F.col("city"))) \
    .withColumn("state",         F.upper(F.trim(F.col("state")))) \
    .withColumn(
        # Remove impossible negative values
        "total_orders",
        F.when(F.col("total_orders") < 0, F.lit(None))
         .otherwise(F.col("total_orders"))
    ) \
    .withColumn(
        "is_valid_email",
        F.col("email").rlike(r"^[^@]+@[^@]+\.[^@]+$")
    ) \
    .withColumn(
        "days_as_customer",
        F.datediff(F.current_date(), F.col("date_joined"))
    ) \
    .withColumn("silver_created_at", F.current_timestamp())

print("Cleaned customers:", cleaned_df.count())

StatementMeta(, 030a17ad-bb76-45df-9df6-4c736de41ce4, 4, Finished, Available, Finished, False)

Cleaned customers: 500


In [3]:
from pyspark.sql import functions as F

# Remove records with missing customer_id
good_df = cleaned_df.filter(F.col("customer_id").isNotNull())

# Silver Lakehouse Delta table path
TARGET_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_customers"
)

# Write as Delta
(
    good_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TARGET_PATH)
)

# Read back for validation
silver_customers = (
    spark.read
    .format("delta")
    .load(TARGET_PATH)
)

# Validate
row_count = silver_customers.count()

print("=" * 60)
print("✅ Silver Customers table created successfully!")
print(f"📂 Path      : {TARGET_PATH}")
print(f"📊 Total Rows: {row_count}")
print("=" * 60)

# Preview data
silver_customers.show(10, truncate=False)

StatementMeta(, 030a17ad-bb76-45df-9df6-4c736de41ce4, 5, Finished, Available, Finished, False)

✅ Silver Customers table created successfully!
📂 Path      : abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Tables/silver_customers
📊 Total Rows: 500
+-----------+----------+---------+---------------------------+-------------+---------+-----+-------+-----------+------------+------------+---------+--------------+----------------+--------------------------+
|customer_id|first_name|last_name|email                      |phone        |city     |state|pincode|date_joined|loyalty_tier|total_orders|is_active|is_valid_email|days_as_customer|silver_created_at         |
+-----------+----------+---------+---------------------------+-------------+---------+-----+-------+-----------+------------+------------+---------+--------------+----------------+--------------------------+
|CUST10000  |Arjun     |Nair     |arjun.nair0@rediffmail.com |+918647547307|Mumbai   |DL   |103801 |2022-11-19 |Platinum    |24          |true     |true          |1321            |